In [4]:
import sys
sys.path.append("/Users/janikeuskirchen/Code/cargopal/cargoformer/venv/lib/python3.8/site-packages")

In [8]:
import numpy as np
from squaternion import Quaternion
from typing import List 

In [6]:
NUM_ITEMS = 10
NUM_SIMULATIONS = 1
ORIENTATION_NOISE = 0.01

In [84]:
def generate_random_item():
    dim = np.random.uniform(0.1, 0.5, size=3).round(4)
    pos = [*np.random.uniform(-3.0, 3.0, size=2).round(4),
           max(np.round(np.random.uniform(0, 3.0), 4), dim[2]/2+0.01)]
    # Making sure that the pos_height is at least the dim_height/2, otherwise the item is stuck in the ground
    # re-order so it's the same ordering as in pybullet; only if I use squaternion above!
    # each item's vector looks like this: [x, y, z, w, d, h, q1, q2, q3, q4, m]
    # [x, y, z] is the position, [w, d, h] is the shape
    # Orientation
    q = np.array(Quaternion.from_euler(0., 0., 0.))
    q = q[[1, 2, 3, 0]]  # re-order so it's the same ordering as in pybullet; only if I use squaternion above!
    # add a little bit of noise to the orientation
    noise = np.random.normal(0, ORIENTATION_NOISE, size=4)
    q = (q + noise).round(4)
    orn = q
    # for now, assume zero orientation (they're all cuboids; I can change orientation by changing its dimensions)
    # for Euler orientation I would use notation [a, b, g] (alpha, beta, gamma)
    # Mass
    # to make things easy, mass is always simply proportional to volume and this proportionality coefficient is
    # randomly sampled to be between, say, 5 and 15, according to a uniform distribution
    m = (np.prod(dim)*np.random.uniform(5, 15)).round(4)
    # Each item's vector looks like this: [x, y, z, w, d, h, q1, q2, q3, q4, m]
    return np.array([*pos, *dim, *orn, m])

In [89]:
def intersects(item1, item2):
    # https://gamedev.stackexchange.com/questions/23748/testing-whether-two-cubes-are-touching-in-space 
    # True if item1 and item2 intersect
    # Note that w, h, d are half-extents!!
    x1, y1, z1, w1, d1, h1, _, _, _, _, _ = item1
    x2, y2, z2, w2, d2, h2, _, _, _, _, _ = item2
    min_x1, max_x1, min_y1, max_y1, min_z1, max_z1 = x1-w1, x1+w1, y1-d1, y1+d1, z1-h1, z1+h1
    min_x2, max_x2, min_y2, max_y2, min_z2, max_z2 = x2-w2, x2+w2, y2-d2, y2+d2, z2-h2, z2+h2
    return ((min_x1 <= min_x2 and min_x2 <= max_x1) or (min_x2 <= min_x1 and min_x1 <= max_x2)) and \
           ((min_y1 <= min_y2 and min_y2 <= max_y1) or (min_y2 <= min_y1 and min_y1 <= max_y2)) and \
           ((min_z1 <= min_z2 and min_z2 <= max_z1) or (min_z2 <= min_z1 and min_z1 <= max_z2))

def any_intersects(item1, items):
    for item in items:
        if intersects(item1, item):
            return True
    return False

In [91]:
def generate_random_layout():
    items = []
    for i in range(100):
        proposed_item = generate_random_item()
        if i > 0:
            while any_intersects(proposed_item, items):
                print("trying again")
                proposed_item = generate_random_item()
        items.append(proposed_item)
        print(proposed_item)
    return items 

items = generate_random_layout()

[ 1.8971e+00 -2.4225e+00  5.1750e-01  3.7180e-01  3.1490e-01  2.9450e-01
 -1.6500e-02  2.0000e-04  1.9000e-03  9.8780e-01  3.5710e-01]
[ 2.5136e+00 -1.7626e+00  2.8108e+00  3.0920e-01  4.0630e-01  3.1030e-01
  5.9000e-03 -9.0000e-03 -5.0000e-04  9.8070e-01  2.2180e-01]
[-1.7675  1.5575  1.36    0.3055  0.3318  0.4581 -0.003  -0.0129 -0.0047
  1.0239  0.3117]
[ 2.6426e+00 -7.6470e-01  1.2100e+00  4.9700e-01  4.3430e-01  3.6530e-01
  9.1000e-03 -2.5000e-03 -1.3300e-02  9.9960e-01  1.0674e+00]
[-2.1244  0.6833  1.0095  0.3948  0.4899  0.1483 -0.0167  0.014  -0.0148
  0.999   0.2431]
[-1.4620e+00  2.4218e+00  1.6263e+00  1.4390e-01  2.4260e-01  4.5930e-01
  4.1000e-03  2.0000e-03 -2.3300e-02  1.0105e+00  2.0190e-01]
[ 1.1768e+00 -6.6780e-01  1.0375e+00  4.7080e-01  2.9980e-01  2.1770e-01
 -9.0000e-04 -1.2900e-02  2.0000e-03  9.8160e-01  4.5160e-01]
[-2.7759 -0.0482  0.9404  0.1468  0.4557  0.1204 -0.0091  0.0041 -0.0157
  0.9982  0.1202]
[-7.7150e-01  1.7763e+00  6.5350e-01  4.5880e-01  4.